# Workshop: Generating a Ventilation Controller from a Knowledge Graph

In this workshop, we'll demonstrate how to use a **Knowledge Graph** to automatically configure a controller application.

**Our Goal:**
1.  Connect to a SPARQL endpoint (like an RDF4J server).
2.  Query the KG to find all rooms (**Query 1**).
3.  For each room, find its ventilation actuators (**Query 2c**).
4.  Apply logic: Check what kind of sensors are available for that room to decide on the best control strategy (**Query 3c**).
    * *Priority 1:* Use a **CO2 sensor** for demand-controlled ventilation.
    * *Priority 2:* Use a **Presence sensor** for occupancy-based control.
    * *Fallback:* Use a simple **Timetable** if no sensors are available.
5.  Generate a YAML configuration file that our controller application can read.

<img src="images/query_workflow.png" alt="Query Workflow" width="400">

## Step 1: Imports necessary libraries and initialize API client

In [14]:
import os
import yaml
from pathlib import Path
from rdf4j_client import SparqlApiClient

graphdb_url = "https://ebc-demo-graphdb.eonerc.rwth-aachen.de/repositories/InteroperabilityWorkshop"
query_client = SparqlApiClient(endpoint_url=graphdb_url, credential_path="credential.json")

SparqlApiClient (requests-based) initialized. Pointing to: https://ebc-demo-graphdb.eonerc.rwth-aachen.de/repositories/InteroperabilityWorkshop


## Step 2: Perform logical inference with Ontology (Brick)
In this step, we will perform logical inference using the Brick ontology to enrich our knowledge graph. This will help us derive additional relationships and properties that are not explicitly stated in the original data.

Go to GraphDB GUI -> Import -> Upload RDF Files -> Select the Brick ontology file


## Step 3: Information retrieval via SPARQL

### Query 1 find existing rooms

In [15]:
query_rooms = """
    PREFIX rec: <https://w3id.org/rec#>
    SELECT DISTINCT ?room
    WHERE {
        ?room a rec:Room
    }
"""
query1_res = query_client.query(query_string=query_rooms)
print(f"Number of rooms found: {len(query1_res)}")
# First room
room_iri = query1_res.bindings[8]["room"]
print(f"First room IRI: {room_iri}")

Number of rooms found: 10
First room IRI: http://example.com/HotelRoom/HotelRoom:room_base_8


### Query 2 find available ventilation device and its controllable datapoint

In [16]:
query_ventilation_devices = f"""
    PREFIX rec:   <https://w3id.org/rec#>
    PREFIX brick: <https://brickschema.org/schema/Brick#>
    PREFIX rdf:   <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

    SELECT ?device ?actuation ?actuation_access
    WHERE {{
      <{room_iri}> a rec:Room .

      ?device a brick:Air_System .
      ?actuation brick:isPointOf ?device .

      OPTIONAL {{
        ?actuation a ?actuation_type ;
                   rdf:value ?actuation_access .
        VALUES ?actuation_type {{ brick:Setpoint brick:Command }}
      }}

      ?actuation (brick:isPointOf|brick:hasLocation|brick:isPartOf)* <{room_iri}> .
    }}
"""
query2_res = query_client.query(query_string=query_ventilation_devices)
print(f"Number of ventilation devices found: {len(query2_res)}")
# get the device, actuation point, and access point
if len(query2_res) == 1:
    device = query2_res.bindings[0].get("device")
    actuation = query2_res.bindings[0].get("actuation")
    actuation_access = query2_res.bindings[0].get("actuation_access")
    print(f"Device IRI: {device}\nActuation IRI: {actuation}\nAPI Endpoint: {actuation_access}")

Number of ventilation devices found: 1
Device IRI: http://example.com/FreshAirVentilation/FreshAirVentilation:room_base_8
Actuation IRI: http://example.com/airFlowSetpoint_FreshAirVentilation/airFlowSetpoint_FreshAirVentilation:room_base_8
API Endpoint: https://fiware.eonerc.rwth-aachen.de/v2/entities/FreshAirVentilation:room_base_8/attrs/airFlowSetpoint/value


### Query Set 3: Check for available sensors in the room
Try to find a CO2 sensor first.

In [17]:
query_co2_sensor_availability = f"""
    PREFIX rec: <https://w3id.org/rec#>
    PREFIX brick: <https://brickschema.org/schema/Brick#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

    SELECT ?sensor ?sensor_access
    WHERE {{
        <{room_iri}> a rec:Room .
        ?sensor a brick:CO2_Sensor .
        ?sensor rdf:value ?sensor_access .
        ?sensor (brick:isPointOf|brick:hasLocation|brick:isPartOf)* <{room_iri}>.
    }}
"""
query3_res = query_client.query(query_string=query_co2_sensor_availability)
if len(query3_res) > 0:
    sensor = query3_res.bindings[0].get("sensor")
    sensor_access = query3_res.bindings[0].get("sensor_access")
    print(f"CO2 Sensor found")
    print(f"Sensor IRI: {sensor}\nAPI Endpoint: {sensor_access}")
else:
    print("No CO2 sensor found.")

No CO2 sensor found.


If no CO2 sensor is found, try to find a presence sensor.

In [18]:
query_presence_sensor_availability = f"""
    PREFIX rec: <https://w3id.org/rec#>
    PREFIX brick: <https://brickschema.org/schema/Brick#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

    SELECT ?sensor ?sensor_access
    WHERE {{
        <{room_iri}> a rec:Room .
        ?sensor a brick:Occupancy_Count_Sensor .
        ?sensor rdf:value ?sensor_access .
        ?sensor (brick:isPointOf|brick:hasLocation|brick:isPartOf)* <{room_iri}>.
    }}
"""
query3_res_presence = query_client.query(query_string=query_presence_sensor_availability)
if len(query3_res_presence) > 0:
    sensor = query3_res_presence.bindings[0].get("sensor")
    sensor_access = query3_res_presence.bindings[0].get("sensor_access")
    print(f"Presence Sensor found")
    print(f"Sensor IRI: {sensor}\nAPI Endpoint: {sensor_access}")
else:
    print("No Presence sensor found.")

No Presence sensor found.


## Step 4: Apply the same logic to the whole dataset and generate configuration file

The same queries are implemented in the `ControllerConfiguration` class.
After querying the knowledge graph, a configuration file in YAML format will be generated for deploying controller applications.

In [19]:
from controller_config import ControllerConfiguration
import yaml

configurator = ControllerConfiguration(
    api_client=query_client,
)
configuration = configurator.generate_configuration()


--- Starting Configuration Generation ---
Found 10 rooms. Processing...

Processing Room: http://example.com/HotelRoom/HotelRoom:room_base_1
  -> Found Actuator: https://fiware.eonerc.rwth-aachen.de/v2/entities/FreshAirVentilation:room_base_1/attrs/airFlowSetpoint/value
  -> Found CO2 Sensor: https://fiware.eonerc.rwth-aachen.de/v2/entities/CO2Sensor:room_base_1/attrs/co2/value

Processing Room: http://example.com/HotelRoom/HotelRoom:room_base_5
  -> Found Actuator: https://fiware.eonerc.rwth-aachen.de/v2/entities/FreshAirVentilation:room_base_5/attrs/airFlowSetpoint/value
  -> Found CO2 Sensor: https://fiware.eonerc.rwth-aachen.de/v2/entities/CO2Sensor:room_base_5/attrs/co2/value

Processing Room: http://example.com/HotelRoom/HotelRoom:room_base_6
  -> Found Actuator: https://fiware.eonerc.rwth-aachen.de/v2/entities/FreshAirVentilation:room_base_6/attrs/airFlowSetpoint/value
  -> Found CO2 Sensor: https://fiware.eonerc.rwth-aachen.de/v2/entities/CO2Sensor:room_base_6/attrs/co2/value


In [20]:
# View the generated configuration
print(f"\n--- Generation Complete ---")
print(f"Configuration for {len(configuration)} controllers:")
print(yaml.dump(configuration, sort_keys=False))


--- Generation Complete ---
Configuration for 10 controllers:
- controller_function: Ventilation
  controller_mode: co2
  inputs:
    sensor_access: https://fiware.eonerc.rwth-aachen.de/v2/entities/CO2Sensor:room_base_1/attrs/co2/value
  outputs:
    actuation_access: https://fiware.eonerc.rwth-aachen.de/v2/entities/FreshAirVentilation:room_base_1/attrs/airFlowSetpoint/value
- controller_function: Ventilation
  controller_mode: co2
  inputs:
    sensor_access: https://fiware.eonerc.rwth-aachen.de/v2/entities/CO2Sensor:room_base_5/attrs/co2/value
  outputs:
    actuation_access: https://fiware.eonerc.rwth-aachen.de/v2/entities/FreshAirVentilation:room_base_5/attrs/airFlowSetpoint/value
- controller_function: Ventilation
  controller_mode: co2
  inputs:
    sensor_access: https://fiware.eonerc.rwth-aachen.de/v2/entities/CO2Sensor:room_base_6/attrs/co2/value
  outputs:
    actuation_access: https://fiware.eonerc.rwth-aachen.de/v2/entities/FreshAirVentilation:room_base_6/attrs/airFlowSetp